# Semana 3 – Arquitectura del Pipeline y Prevención de Data Leakage

Objetivos:
- Diseñar pipelines reproducibles
- Evitar fuga de datos
- Comparar técnicas de escalamiento
- Evaluar impacto arquitectónico en producción

# Carga y Exploración

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


# Análisis Básico

In [10]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         5

,0
mean radius,0
mean texture,0
mean perimeter,0
mean area,0
mean smoothness,0
mean compactness,0
mean concavity,0
mean concave points,0
mean symmetry,0
mean fractal dimension,0


**ProfileReport** que permite realizar un Análisis Exploratorio de Datos (EDA) completo y automatizado.

**¿Qué incluye el reporte?**


In [13]:

!pip install ydata-profiling
from ydata_profiling import ProfileReport
# Generar reporte
profile = ProfileReport(df,
                        title="Reporte Exploratorio - Breast Cancer Dataset",
                        explorative=True)

# Guardar en archivo HTML
profile.to_file("reporte_breast_cancer.html")

# Mostrar en notebook
profile



Output hidden; open in https://colab.research.google.com to view.

In [14]:
profile = ProfileReport(df, minimal=True)
profile.to_file("reporte_minimal.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 31/31 [00:00<00:00, 106.37it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# Detección de Baja Varianza

**VarianceThreshold** es un enfoque básico simple para la selección de características. Elimina todas las características cuya varianza no alcanza un umbral determinado. De forma predeterminada, elimina todas las características con varianza cero, es decir, aquellas que tienen el mismo valor en todas las muestras.
https://www.geeksforgeeks.org/machine-learning/variance-threshold/

In [4]:
from sklearn.feature_selection import VarianceThreshold

X = df.drop('target', axis=1)
selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(X)

print("Variables originales:", X.shape[1])
print("Variables después del filtro:", X_var.shape[1])

Variables originales: 30
Variables después del filtro: 14


**Discusión:**

¿Qué implica eliminar baja varianza?

¿Impacto en interpretabilidad?

# Experimento 1: Enfoque Incorrecto (Data Leakage)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df.drop('target', axis=1)
y = df['target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # ❌ ERROR conceptual

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)


0.9824561403508771

**Reflexión:**

¿Dónde ocurrió la fuga?

¿Por qué es arquitectónicamente incorrecto?

# Enfoque Correcto con Pipeline
pipeline (o canalización) es una serie de elementos de procesamiento de datos conectados en serie, donde la salida de un elemento es la entrada del siguiente. Se utiliza para automatizar el flujo de información, transformando datos en bruto de diversas fuentes para almacenarlos o analizarlos, permitiendo eficiencia y el trabajo paralelo en informática
https://www.ibm.com/mx-es/think/topics/machine-learning-pipeline

**StratifiedKFold**
es una variante de la técnica de validación cruzada K-Fold diseñada específicamente para problemas de clasificación. Su función principal es dividir un conjunto de datos en
partes (o "pliegues") manteniendo la misma proporción de clases en cada una de ellas que la que existe en el dataset original.

**Distribución de clases:** A diferencia del K-Fold estándar, que divide los datos de forma aleatoria sin considerar las etiquetas, StratifiedKFold garantiza que si una clase representa el 10% del total, también representará aproximadamente el 10% en cada pliegue de entrenamiento y prueba.

**Ideal para datos desbalanceados:** Es indispensable cuando algunas clases son mucho más frecuentes que otras. Sin la estratificación, un pliegue de prueba podría terminar con cero muestras de una clase minoritaria, lo que daría una evaluación sesgada del modelo.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')

print("Accuracy promedio:", scores.mean())
print("Desviación estándar:", scores.std())


Accuracy promedio: 0.9736686849868033
Desviación estándar: 0.016627222216787987


## Comparativa: KFold vs. StratifiedKFold

| Característica | KFold Estándar | StratifiedKFold |
|----------------|----------------|-----------------|
| División | Aleatoria | Conserva proporciones de clase |
| Uso principal | Regresión o clasificación con clases equilibradas | Clasificación (especialmente con clases desbalanceadas) |
| Riesgo | Puede generar pliegues sin representación de clases raras | Garantiza representatividad de cada clase en cada iteración |


**Discusión:**

Ventajas de encapsular transformaciones

Reproducibilidad

Producción

# Comparación de Escaladores

| Característica      | StandardScaler      | MinMaxScaler                  | RobustScaler       |
| ------------------- | ------------------- | ----------------------------- | ------------------ |
| Centrado            | Media               | No necesariamente             | Mediana            |
| Escalado por        | Desviación estándar | Rango (min-max)               | IQR                |
| Rango final         | No fijo             | Fijo (ej. 0–1)                | No fijo            |
| Sensible a outliers | Sí                  | Sí (mucho)                    | No (mucho menos)   |
| Ideal para          | Datos normales      | Redes neuronales / rango fijo | Datos con outliers |


In [7]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler()
}

for name, scaler in scalers.items():
    pipe = Pipeline([
        ('scaler', scaler),
        ('model', LogisticRegression(max_iter=5000))
    ])

    scores = cross_val_score(pipe, X, y, cv=5)
    print(f"{name}: {scores.mean():.4f}")


StandardScaler: 0.9807
MinMaxScaler: 0.9613
RobustScaler: 0.9789


**Análisis:**

¿Cuál fue más estable?

¿Qué sucede con outliers?

1️⃣ StandardScaler

🔹 ¿Qué hace?

Estandariza los datos eliminando la media y escalando a varianza unitaria.

🔹 Cuándo usarlo

Modelos que asumen distribución aproximadamente normal.

Algoritmos sensibles a escala:

*   SVM
*   KNN
*   Redes neuronales
*   Regresión logística
*   PCA

🔹 Problema

Muy sensible a outliers, porque usa media y desviación estándar.

2️⃣ MinMaxScaler

🔹 ¿Qué hace?

Escala los datos a un rango específico (por defecto [0,1]).

🔹 Resultado

Todos los valores quedan entre 0 y 1 (o el rango definido).

🔹 Cuándo usarlo

* Redes neuronales (cuando se usan activaciones como sigmoid).

* Datos que necesitan mantenerse en un rango acotado.

* Cuando la distribución no importa tanto como el rango.

🔹 Problema

Extremadamente sensible a outliers, porque usa valores mínimo y máximo.

3️⃣ RobustScaler

🔹 ¿Qué hace?

Escala usando estadísticas robustas donde:

Mediana en lugar de media

IQR = rango intercuartílico (Q3 - Q1)

🔹 Cuándo usarlo

* Datos con muchos outliers.

* Distribuciones no normales.

* Datos financieros, sensores, variables con colas largas.

🔹 Ventaja clave

Mucho menos sensible a valores extremos.